# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading and exploring a dataset using the `mlcroissant` library. We will walk step-by-step through loading the dataset defined by a FAIR Croissant schema, inspecting its metadata, extracting data using `@id` for records and fields, and performing exploratory analysis and simple visualizations.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

We load the dataset metadata (and later records) using the `mlcroissant` library. The Croissant schema URL uniquely identifies the dataset and its structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset: ", metadata.name)
print("Description: ", metadata.description)
print("Identifier:", getattr(metadata, 'identifier', None))
print("Version:", getattr(metadata, 'version', None))


## 2. Data Overview

Let's list the available record sets, their corresponding `@id` values, and inspect which fields are available in each. This ensures all subsequent data extraction and analysis steps reference entities by their Croissant `@id`.

In [ ]:
# List all record sets and their fields (by @id)
rs = list(dataset.metadata.record_sets or [])

if not rs:
    print("No record sets found in this Croissant package. Exiting.")
else:
    for rset in rs:
        print(f"RecordSet name: {getattr(rset, 'name', None)}\n  @id: {rset.id}\n  Description: {getattr(rset, 'description', 'No description')}")
        print("  Fields:")
        for field in (rset.fields or []):
            print(f"    - {getattr(field, 'name', field.id)} (@id: {field.id}, dtype: {getattr(field, 'data_type', 'Unknown')})")
        print()


## 3. Data Extraction

We'll extract data from record sets using their `@id`. In this dataset, there may be only one main record set containing the protein data. We will identify the `@id`, fields, and show the DataFrame columns after loading a sample.

In [ ]:
# For demonstration, we'll auto-pick the main record set (first one). Adapt the variable below if there are multiple.
rs = list(dataset.metadata.record_sets or [])
if not rs:
    raise ValueError("No record sets available.")

main_record_set = rs[0]
main_record_set_id = main_record_set.id
print(f"Main RecordSet selected: {main_record_set_id}")
# If there are multiple, you can adjust 'record_set_ids' below accordingly.

record_set_ids = [r.id for r in rs]

dataframes = {}
for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        dataframes[rset_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {rset_id} with {len(dataframes[rset_id])} rows.")
    except Exception as e:
        print(f"Failed loading data for {rset_id}: {e}")

# Show column names of the main DataFrame
df = dataframes[main_record_set_id]
print("Columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field and perform basic filtering, normalization, and grouping. All field access is done using `@id` as keys for columns. We'll list available numeric fields so you can adjust variables as needed for further exploration.

In [ ]:
# List numeric fields in the main record set
from collections.abc import Iterable
numeric_fields = [
    field.id for field in (main_record_set.fields or [])
    if getattr(field, 'data_type', '').lower() in ['float', 'number', 'integer']
]
print("Available numeric fields (by @id):\n", numeric_fields)

# Select a numeric field for demonstration (adjust as desired)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # Fall back to first available field
    numeric_field_id = df.columns[0] if len(df.columns) > 0 else None

print(f"Using numeric field: {numeric_field_id}")

# Basic filtering
threshold = df[numeric_field_id].dropna().mean() if numeric_field_id and numeric_field_id in df else 0
filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id in df else df
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalizing numeric field
if numeric_field_id in filtered_df:
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, norm_col]].head())
else:
    print(f"Field {numeric_field_id} not found in the dataframe.")

# Group by a categorical field if available
group_fields = [field.id for field in (main_record_set.fields or []) if getattr(field, 'data_type', '').lower() in ['string', 'text']]
if group_fields:
    group_field_id = group_fields[0]
    print(f"Grouping by {group_field_id}...")
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(grouped_df.head())
else:
    print("No categorical fields available for grouping.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its normalized version (if any).

In [ ]:
# Plot distribution of numeric field (raw and normalized)
if numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].fillna(0).hist(bins=30, alpha=0.7, label=numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.legend()
    plt.show()

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(7, 4))
        filtered_df[f"{numeric_field_id}_normalized"].dropna().hist(bins=30, alpha=0.7, color='g', label=f"{numeric_field_id}_normalized")
        plt.title(f'Normalized Distribution of {numeric_field_id}')
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel('Count')
        plt.legend()
        plt.show()

## 6. Conclusion

- We loaded and inspected the dataset metadata via Croissant with `mlcroissant`.
- All exploration referenced entities (record sets, fields, columns) by `@id` as defined by the schema.
- Data extraction, filtering, normalization, and basic visualizations were performed on the main protein record set.

You can now adapt the code for more advanced analysis, using additional fields or further processing based on the rich Croissant metadata.